# FAβ-Gal - Fluorescence Analysis of senescence-associated β-Gal 

This Jupyter Notebook is designed to guide the user through the analysis of fluorescence images of cell culture or tissue sections senescence-associated beta-galactosidase (SA-βgal) assays, enabling a quantitative approach to this routine technique in aging studies.

The FAβ-Gal workflow first loads user images located in the specified **input_folder**. Input images must be in TIFF format (.tif or .tiff), and be taken using the same objective, laser intensity and time exposure for the far-red channel (βgal channel). 


## Options

The FAβ-Gal pipeline has two optional actions:

- **delete_intermediate_files**:\
When set to `True` (default), removes intermediate files generated by BiaPy at the end of the pipeline execution. These files can be useful if you are adjusting BiaPy parameters.

- **substract_background**:\
When set to `True` (default), performs the substract background algorithm nuceli channel of input images. This algorithm reduces the background fluorescence enhacing Biapy nuclei segmentation so we strongly recommend it. The radius parameter of the algorithm can be modified, because it should be bigger than the radius of the smallest object that you want to segmentate, hence, bigger than the radius of the nuclei. For more details you can check the code function in **function.py**




In [ ]:
delete_intermediate_files = True
substract_background = True 

## Parameters

FAβ-Gal pipeline has the following parameters:

- **input_folder**:\
Input images directory (Must exist).

- **experiment_name**:\
Experiment name for naming output files generated in the analysis.

- **img_to_ind**:\
Tab-separated file with 2 columns: File (filenames of images) and Individual (biological replicate, eg. well, to which the image belongs). This file is used to compute the CTF per nuceli (for culture cell analyses) not only for each image but also at the biological replicate level (e.g. well), putting together the data of all the technical replicates (e.g. images of the same well). We strongly recommend this because it mitigates the effect of possible outliers such as images with a low number of cells. When set to `None` (default) FAβ-Gal only calculates the CTF per nuclei for aech image.


<div align="center">

**Example `img_to_ind.tsv`**

| File           | Individual |
|----------------|------------|
| image_001.tif  | A01        |
| image_002.tif  | A01        |
| image_003.tif  | A02        |
| image_004.tif  | B01        |
| image_005.tif  | B01        |

</div>

- **nuclei_ch**:\
Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0).

- **bgal_ch**:\
Channel containing BGal stain signal.

- **pixel_size**:\
If pixel size is to be entered manually, change it here. Defaults to None, where the software will get the info from the image or, if no info is found, use 1 as default

- **nuclei_ch**:\
Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0)

- **nuclei_ch**:\
Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0)

In [ ]:

# Files
input_folder="./individuals/" 
experiment_name = "Experiment1" 
img_to_ind = None 
# Image parameters
nuclei_ch = 0 # Channel containing nuclei stain signal. Channels follow a 0-based numbering (first channel is 0).
bgal_ch = 1 # Channel containing BGal stain signal.
pixel_size = None # If pixel size is to be entered manually, change it here. Defaults to None, where the software will get the info from the image or, if no info is found, use 1 as default

# B-Gal background parameters
backgr_val = 15 # Mean intensity of B-Gal channel in a background image obtained elsewhere to be entered manually. If None, the software will calculate it using backgr_img.
backgr_img = None # Name of an image from which the background mean intensity will be calculated if no backgr_val is entered.
# If both backgr_val and backgr_img are None, neither CTF/nuclei nor CTF/area will be calculated.

# Bgal parameters
bgal_th = 45 # Minimum value of BGal stain signal to be considered positive. Default: 45.

# Substract background parameters (for nuclei)
sbg_rad = 31 # Radius of substract background. Should be adjusted depending on nuclei size. Default: 31.

# Biapy parameters
config_path = "./instance_segmentation.yaml"            # Path to BiaPy YAML configuration file
result_dir = "./biapy_output"                 # Directory to store BiaPy results
job_name = experiment_name                      # Name of BiaPy job. Dafaults to experiment name.
run_id = 1                                      # Run ID for logging/versioning
gpu = "0"                                       # GPU to use (as string, e.g., "0"). Obtained from nvidia-smi command.

## Measure B-Gal and create tiffs for biapy


In [32]:
from bioio import BioImage
from bioio_base.exceptions import UnsupportedFileFormatError
import os
from re import sub
from bioio_ome_tiff.writers import OmeTiffWriter
from functions import calculate_bgal, subtract_background  # Local file with B-Gal quantification and substract background functions.

# Check if biapy_input folder exists, if not create it
if not os.path.isdir("./biapy_input"):
    os.makedirs("./biapy_input")

# Read all files
try:
    myfiles = os.listdir(input_folder)
except FileNotFoundError as e:
    raise FileNotFoundError("ERROR: Input directory does not exist. Please check input_folder variable.") from e

myfiles = [f for f in myfiles if f.lower().endswith(('.tif', '.tiff'))]


# Iterate over all files
bres = {}
for inf in myfiles:
    print("Doing file " + inf)

    try:
        img = BioImage(os.path.join(input_folder, inf))
    except UnsupportedFileFormatError as e:
        raise UnsupportedFileFormatError("\nERROR: Image is not in TIFF format.") from e
    except Exception as e:
        raise Exception(f"\nERROR: Could not open {inf}: {e}") from e
    
    # Measure bgal area
    bres[sub(r'(?i)\.tiff?$', '', inf)] = calculate_bgal(img,bgal_ch,bgal_th,psize=pixel_size)

    # Save tiff for biapy input
    if(substract_background):
        OmeTiffWriter.save(subtract_background(img.get_image_data("YX",C=nuclei_ch), radius=sbg_rad), 
                    "./biapy_input/" + inf, dim_order="YX")
    else:
       OmeTiffWriter.save(img.get_image_data("YX",C=nuclei_ch), 
                    "./biapy_input/" + inf, dim_order="YX")


# Save bgal results
bgal_f = open(experiment_name + "_Raw_BGal_results.tsv",'w')
bgal_f.write("File\tNpxPos\tNpxTot\tAreaPos\tAreaTot\tBgal_RawIntDen\n")
for k, (npx, npxtot, areapos, areatot, RawIntDen) in bres.items():
    bgal_f.write(f"{k}\t{npx}\t{npxtot}\t{areapos}\t{areatot}\t{RawIntDen}\n")
bgal_f.close()

[18:27:24.356266] Doing file 20250905_Plate_seeded_20250902_BGal-02-Scene-12-P4-E06.tif
[18:27:24.384127] Image pixel sizes are PhysicalPixelSizes(Z=None, Y=0.5476193344672704, X=0.5476193344672704)
[18:27:25.039303] Doing file Untitled.ome.tif
[18:27:25.059915] Image pixel sizes are PhysicalPixelSizes(Z=None, Y=0.5476190476190477, X=0.5476190476190477)


## Run Biapy

In [33]:
# Create and run the BiaPy job

from biapy import BiaPy
biapy = BiaPy(config_path, result_dir=result_dir, name=job_name, run_id=run_id, gpu=gpu)
biapy.run_job()

[18:27:25.715554] Date     : 2026-01-22 18:27:25
[18:27:25.715598] Arguments: Namespace(config='./instance_segmentation.yaml', result_dir='./biapy_output', name='Experiment1', run_id=1, gpu='0', world_size=1, local_rank=-1, dist_on_itp=False, dist_url='env://', dist_backend='nccl')
[18:27:25.715606] Job      : Experiment1_1
[18:27:25.715613] BiaPy    : 3.6.8
[18:27:25.715619] Python   : 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:46:49) [Clang 19.1.7 ]
[18:27:25.715630] PyTorch  : 2.9.1
[18:27:25.722424] The following changes were made in order to adapt the input configuration:
[18:27:25.724590] Not using distributed mode
[18:27:25.728008] WARNING: seems that the channels requested are custom so BiaPy did not fill some varibles by default.
You will need to fill the following variables:
    - PROBLEM.INSTANCE_SEG.WATERSHED.SEED_CHANNELS
    - PROBLEM.INSTANCE_SEG.WATERSHED.SEED_CHANNELS_THRESH
    - PROBLEM.INSTANCE_SEG.WATERSHED.TOPOGRAPHIC_SURFACE_CHANNEL
    - PROBLEM.

[18:27:56.692256] ### MERGE-OV-CROP ###
[18:27:56.692291] Merging (143, 256, 256, 2) images into (1, 2056, 2464, 2) with overlapping . . .
[18:27:56.692301] Minimum overlap selected: (0, 0)
[18:27:56.692317] Padding: (32, 32)
[18:27:56.695630] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[18:27:56.695651] Real overlapping (pixels): (2.0, 5.0)
[18:27:56.695659] (13, 11) patches per (x,y) axis
[18:27:56.753207] **** New data shape is: (1, 2056, 2464, 2)
[18:27:56.753238] ### END MERGE-OV-CROP ###
[18:27:56.754613] ### MERGE-OV-CROP ###
[18:27:56.754624] Merging (143, 256, 256, 1) images into (1, 2056, 2464, 1) with overlapping . . .
[18:27:56.754631] Minimum overlap selected: (0, 0)
[18:27:56.754636] Padding: (32, 32)
[18:27:56.757043] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[18:27:56.757055] Real overlapping (pixels): (2.0, 5.0)
[18:27:56.757063] (13, 11) patches per (x,y) axis
[18:27:56.781633] **** New data shape is: (1, 2056, 2464, 1)


[18:27:56.880915] Creating instances with watershed . . .
[18:27:57.023332] Thresholds used: {'seed': [0.7, 0.6], 'growth_mask': [0.23729699850082397]}
[18:27:57.270537] Saving (1, 1, 2056, 2464, 1) data as .tif in folder: ./biapy_output/Experiment1/results/Experiment1_1/per_image_instances


[18:27:57.283443] Checking the properties of instances . . .


[18:27:57.456461] Removed 0 instances by properties ([]), 631 instances left


[18:27:57.494628] Processing image: Untitled.ome.tif
[18:27:57.496006] ### OV-CROP ###
[18:27:57.496029] Cropping (1, 2056, 2464, 1) images into (256, 256, 1) with overlapping. . .
[18:27:57.496037] Minimum overlap selected: (0, 0)
[18:27:57.496043] Padding: (32, 32)
[18:27:57.499370] Real overlapping (%): 0.010416666666666666
[18:27:57.499387] Real overlapping (pixels): 2.0
[18:27:57.499393] 13 patches per (x,y) axis
[18:27:57.505331] **** New data shape is: (143, 256, 256, 1)
[18:27:57.505340] ### END OV-CROP ###


[18:28:14.620964] ### MERGE-OV-CROP ###
[18:28:14.620994] Merging (143, 256, 256, 2) images into (1, 2056, 2464, 2) with overlapping . . .
[18:28:14.621002] Minimum overlap selected: (0, 0)
[18:28:14.621014] Padding: (32, 32)
[18:28:14.625259] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[18:28:14.625272] Real overlapping (pixels): (2.0, 5.0)
[18:28:14.625278] (13, 11) patches per (x,y) axis
[18:28:14.674670] **** New data shape is: (1, 2056, 2464, 2)
[18:28:14.674715] ### END MERGE-OV-CROP ###
[18:28:14.676103] ### MERGE-OV-CROP ###
[18:28:14.676115] Merging (143, 256, 256, 1) images into (1, 2056, 2464, 1) with overlapping . . .
[18:28:14.676122] Minimum overlap selected: (0, 0)
[18:28:14.676127] Padding: (32, 32)
[18:28:14.678926] Real overlapping (%): (0.010416666666666666, 0.026041666666666668)
[18:28:14.678939] Real overlapping (pixels): (2.0, 5.0)
[18:28:14.678945] (13, 11) patches per (x,y) axis
[18:28:14.701494] **** New data shape is: (1, 2056, 2464, 1)


[18:28:14.791003] Creating instances with watershed . . .
[18:28:14.939575] Thresholds used: {'seed': [0.7, 0.6], 'growth_mask': [0.23729699850082397]}
[18:28:15.154780] Saving (1, 1, 2056, 2464, 1) data as .tif in folder: ./biapy_output/Experiment1/results/Experiment1_1/per_image_instances


[18:28:15.414403] Checking the properties of instances . . .


[18:28:15.588523] Removed 0 instances by properties ([]), 631 instances left
[18:28:15.598750] Releasing memory . . .
[18:28:15.598782] #############
[18:28:15.598789] #  RESULTS  #
[18:28:15.598794] #############
[18:28:15.598799] The values below represent the averages across all test samples
[18:28:15.598818] Instance segmentation specific metrics:
[18:28:15.598838] FINISHED JOB Experiment1_1 !!


## Process BiaPy's output

In [34]:
import pandas as pd
from skimage import io
from skimage.color import label2rgb
import numpy as np
import os
import shutil


# Biapy's output folder
mydir=os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image_instances")

# Nuclei stats output file
biapystatsfile = experiment_name + "_BiaPy_results.tsv"

### Process BiaPy output images

Read all label images, convert then to RGB, save as png and delete original tif

In [35]:
myfiles = os.listdir(mydir)
myfiles = [f for f in myfiles if f.endswith(".tif") ]

for inf in myfiles:
    img = io.imread(os.path.join(mydir,inf))
    img = label2rgb(img)*255
    img = img.astype(np.uint8)
    io.imsave(os.path.join(mydir,inf.replace('.tif','.png')),img)
    os.remove(os.path.join(mydir,inf))

### Process nuclei stats

Read all CSVs, add file name, concatenate, save and delete original

In [36]:
myfiles = os.listdir(mydir)
myfiles = [f for f in myfiles if f.endswith("full_stats.csv") ]

#list of csv with file name column
dfs = []
for inf in myfiles:
    df = pd.read_csv(os.path.join(mydir,inf))
    df["File"] = sub("_full_stats.csv","",inf)
    dfs.append(df)

#concatenate
result = pd.concat(dfs, ignore_index=True)
result = result.drop(columns=["conditions"])
result.to_csv(biapystatsfile, sep="\t")

# Delete original
for inf in myfiles:
    os.remove(os.path.join(mydir,inf))

# Merge nuclei and BGal stats

In [37]:
# Load BGal and nuclei stats

nucleidf = pd.read_table(biapystatsfile)
bgaldf = pd.read_table(experiment_name + "_Raw_BGal_results.tsv")

nucleidf
bgaldf


,File,NpxPos,NpxTot,AreaPos,AreaTot,Bgal_RawIntDen
0,20250905_Plate_seeded_20250902_BGal-02-Scene-1...,5065984,378581515,1.519222e+06,1.135317e+08,378581515
1,Untitled.ome,5065984,378581515,1.519221e+06,1.135315e+08,378581515


In [38]:
# Load BGal and nuclei stats

nucleidf = pd.read_table(biapystatsfile)
bgaldf = pd.read_table(experiment_name + "_Raw_BGal_results.tsv")

# Count nuclei per image file 
nucleitot = nucleidf['File'].value_counts()

# B-Gal background intensity
computeCTF = True
if backgr_val is not None:
    BGal_backgr = backgr_val
elif backgr_img is not None:
    BGal_backgr = bgaldf.at[backgr_img, 'BGal_RawIntDen'] / bgaldf.at[backgr_img, 'NpxTot']
else:
    computeCTF = False

# Merge nuclei and B-Gal data
resdf = pd.merge(bgaldf,nucleitot,how='outer',on='File')
resdf = resdf.rename(columns = {'count':'NumNucl'})

# Calculate CTF on individual images (only if background intensity is present)
if computeCTF:
    CTFimg = resdf
    CTFimg['bgMF'] = BGal_backgr
    CTFimg['CTFnucl'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NumNucl
    CTFimg['CTFpix'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NpxTot
    CTFimg['CTFarea'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.AreaTot

    CTFimg.to_csv(experiment_name + "_results_perimage.tsv")

    # Load individual info (if present)
    if img_to_ind is not None:
        resdf = pd.merge(resdf,img_to_ind,how='outer',on='File')

        CTFind = resdf.groupby(by=Individual).sum(numeric_only = True)
        CTFind['bgMF'] = BGal_backgr
        CTFind['CTFnucl'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NumNucl
        CTFind['CTFpix'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.NpxTot
        CTFind['CTFarea'] = (CTFimg.Bgal_RawIntDen - CTFimg.NpxPos * CTFimg.bgMF) / CTFimg.AreaTot

        CTFind.to_csv(experiment_name + "_results_perindividual.tsv")

### Delete extra files generated by Biapy (optional)

Delete intermediate files and files not used in this analysis

In [39]:
if delete_intermediate_files:
    interf = os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image")
    extraf = os.path.join(result_dir,job_name,"results",job_name + "_" + str(run_id),"per_image_post_processing")

    shutil.rmtree(interf)
    shutil.rmtree(extraf)